## Merge Author Profiles — One-ORCID-Anchor, Size-3 to Size-9 Groups

Phase 3b of the author-merge initiative. Extends the rich-name + signal recipe to **size-3 through size-9 groups containing exactly one ORCID-bearing profile** (the "anchor") and 2–8 no-ORCID stragglers. The anchor wins automatically; stragglers merge into it.

Companion to `MergeAuthorsOrcidCollision` (Phase 1) and `MergeAuthorsRichName` (Phase 2). Phase 1 collapsed profiles already sharing an ORCID; Phase 2 handled size-2 same-name pairs (both-no-ORCID + one-ORCID-anchor). This pass picks up the size-3+ extension of the one-ORCID-anchor case — situations where an ORCID-bearing canonical profile has accumulated multiple no-ORCID splinter profiles for the same human.

**Scope:** size-3 through size-9 groups, exactly one ORCID-bearing profile per group. Audit (2026-04-30): ~99% sampled precision (49/50 clean, 0 clear false positives, 1 HEP-flavored marginal) on a 50-group stratified sample with all guards in place.

**Volume estimate:** ~36,854 candidate groups → ~101,217 loser profiles.

**Precision guards:**
1. **Tightened rich-name only.** Dominant parsed `first ≥ 2`, `last ≥ 2`, AND parsed `middle` must contain at least one word ≥ 3 chars (not just total length ≥ 3 — that admits initials-only middles like `B. D.` which collapse the precision lever).
2. **Dominant raw_name majority (≥ 50%).** Per profile, the dominant `raw_author_name` is weighted by `work_authors` row count and must represent ≥ 50% of the profile's `work_authors` rows. Same as Phase 2 — drops profiles with stray contaminating raw_names that compete with the canonical.
3. **Full_name self-consistency.** For every profile in the group, the parsed `last` and the first ≥ 3-char word of parsed `middle` must each appear as whole tokens in the lowercased + ASCII-folded `openalex_authors.full_name`. Catches the dominant remaining false-positive class — profiles whose `full_name` says they're about person X but whose attributions are dominated by person Y (e.g., anchor `5091326369` with `full_name = "Alexandre Silva Vieira"` but dom_raw `"Ana Karine Vieira"`).
4. **Group size 3–9.** Tier-1 size-2 was Phase 2; size-10+ has unique false-positive risk (HEP author-list pollution, weak anchors over collapsed-parser names) and is excluded.
5. **Sink cap.** Drop groups with any profile `works_count > 5,000` (matches Phase 1 / 2 — pollution-sink avoidance).
6. **Exactly one ORCID-bearing profile per group.** Both the count of ORCID-bearing profiles AND the count of distinct ORCIDs equal 1. Guarantees a clean ORCID anchor.
7. **Per-pair precision signal required:** every `(anchor, straggler)` pair must independently share at least one of —
   - **Institution overlap:** any institution_id common to both profiles' `last_known_institutions ∪ affiliations`.
   - **Coauthor overlap ≥ 1:** any other `author_id` co-listed with both pair members on at least one work apiece (excluding other group members from the coauthor count).
   
   *All* stragglers in a group must pass this signal vs the anchor — transitive signal through other stragglers does not count.

**Winner selection:** the ORCID-bearing anchor wins, by construction. (Tiebreaker fields kept in audit log for parity with Phase 2 winner-selection block.)

**Loser handling:** *No DELETE.* Same MERGE-then-soft-zero-via-CreateAuthors approach as Phase 1 (post-reversal) and Phase 2. `work_authors.author_id` rewrites loser → winner; CreateAuthors drains loser `works_count` to 0 on next refresh; Elasticsearch sees an updated zero-works document instead of a missing-id orphan.

**Rollback plan:** No automatic rollback. Audit log `openalex.authors.author_merge_log_orcid_anchor_multi` captures the anchor's ORCID, group-key, and signal flags; sufficient to reconstruct any loser via `STILL_UNMATCHED` + `MatchAuthors`.


### Cell 1: Build merge candidates table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.authors.merge_candidates_orcid_anchor_multi AS
WITH
all_profiles AS (
  SELECT id AS author_id, orcid, full_name, display_name, block_key,
         works_count, cited_by_count, created_date,
         TRANSFORM(COALESCE(last_known_institutions, ARRAY()), x -> x.id) AS lki_ids,
         TRANSFORM(COALESCE(affiliations,            ARRAY()), x -> x.institution.id) AS aff_ids
  FROM openalex.authors.openalex_authors
),
ra_counts AS (
  SELECT wa.author_id, wa.raw_author_name, COUNT(*) AS n_works
  FROM openalex.works.work_authors wa
  JOIN all_profiles n ON n.author_id = wa.author_id
  WHERE wa.raw_author_name IS NOT NULL AND TRIM(wa.raw_author_name) <> ''
  GROUP BY wa.author_id, wa.raw_author_name
),
ra_dominant AS (
  -- Dominant raw_name per profile, weighted by work_authors row count.
  -- Require >= 50% majority to drop profiles where stray contaminating raw_names
  -- compete with the canonical (the Rappard pollution case from Phase 1).
  SELECT author_id, raw_author_name
  FROM (
    SELECT author_id, raw_author_name, n_works,
           SUM(n_works) OVER (PARTITION BY author_id) AS total_works,
           ROW_NUMBER() OVER (PARTITION BY author_id
                              ORDER BY n_works DESC, raw_author_name ASC) AS rn
    FROM ra_counts
  )
  WHERE rn = 1 AND n_works * 2 >= total_works
),
parsed_dominant AS (
  -- Resolve dominant raw_name to (first, middle, last); tightened rich-name filter:
  -- middle must contain at least one word of length >= 3 (not just total length >= 3,
  -- which admits initials-only middles like "B. D." that collapse the precision lever).
  SELECT rd.author_id,
         REGEXP_REPLACE(an.parsed_name.first,  '[.]$', '') AS p_first,
         REGEXP_REPLACE(an.parsed_name.middle, '[.]$', '') AS p_middle,
         REGEXP_REPLACE(an.parsed_name.last,   '[.]$', '') AS p_last
  FROM ra_dominant rd
  JOIN openalex.authors.author_names an ON an.raw_author_name = rd.raw_author_name
  WHERE LENGTH(an.parsed_name.first) >= 2
    AND LENGTH(an.parsed_name.last)  >= 2
    AND SIZE(FILTER(
          SPLIT(REGEXP_REPLACE(an.parsed_name.middle, '[.]', ''), ' +'),
          x -> LENGTH(x) >= 3
        )) >= 1
),
profile_with_dom AS (
  SELECT n.*, p.p_first, p.p_middle, p.p_last,
         (n.orcid IS NOT NULL AND n.orcid <> '') AS has_orcid
  FROM all_profiles    n
  JOIN parsed_dominant p ON p.author_id = n.author_id
),
profile_consistent AS (
  -- Full_name self-consistency guard. Catches profiles whose openalex_authors.full_name
  -- disagrees with the parsed dominant raw_name -- the dominant remaining false-positive
  -- class (anchor 5091326369 = "Alexandre Silva Vieira" but dom_raw = "Ana Karine Vieira"
  -- is the canonical example).
  SELECT pwd.*,
    CASE
      WHEN full_name IS NULL THEN FALSE
      WHEN INSTR(
        CONCAT(' ', LOWER(REGEXP_REPLACE(
          TRANSLATE(full_name,
            'áéíóúüñçÁÉÍÓÚÜÑÇàèìòùâêîôûÀÈÌÒÙÂÊÎÔÛãõÃÕıİğĞşŞçÇöÖüÜæøłÆØŁčćžšŠ',
            'aeiouuncAEIOUUNCaeiouaeiouAEIOUAEIOUaoaoiIgGsScCoOuUaolAOLccszS'),
          '[^a-zA-Z]', ' ')), ' '),
        CONCAT(' ', p_last, ' ')
      ) = 0 THEN FALSE
      WHEN INSTR(
        CONCAT(' ', LOWER(REGEXP_REPLACE(
          TRANSLATE(full_name,
            'áéíóúüñçÁÉÍÓÚÜÑÇàèìòùâêîôûÀÈÌÒÙÂÊÎÔÛãõÃÕıİğĞşŞçÇöÖüÜæøłÆØŁčćžšŠ',
            'aeiouuncAEIOUUNCaeiouaeiouAEIOUAEIOUaoaoiIgGsScCoOuUaolAOLccszS'),
          '[^a-zA-Z]', ' ')), ' '),
        CONCAT(' ',
               FILTER(SPLIT(REGEXP_REPLACE(p_middle, '[.]', ''), ' +'),
                      x -> LENGTH(x) >= 3)[0],
               ' ')
      ) = 0 THEN FALSE
      ELSE TRUE
    END AS full_name_ok
  FROM profile_with_dom pwd
),
group_stats AS (
  -- Size 3-9, exactly one ORCID-bearing profile (the anchor), sink cap, and
  -- ALL profiles in the group pass full_name_ok.
  SELECT p_first, p_middle, p_last
  FROM profile_consistent
  GROUP BY p_first, p_middle, p_last
  HAVING COUNT(*) BETWEEN 3 AND 9
     AND MAX(works_count) <= 5000
     AND COUNT(CASE WHEN has_orcid THEN 1 END) = 1
     AND COUNT(DISTINCT CASE WHEN has_orcid THEN orcid END) = 1
     AND COUNT(*) = COUNT(CASE WHEN full_name_ok THEN 1 END)
),
group_profiles AS (
  SELECT pc.*
  FROM profile_consistent pc
  JOIN group_stats g
    ON pc.p_first = g.p_first AND pc.p_middle = g.p_middle AND pc.p_last = g.p_last
),
anchor AS (
  SELECT p_first, p_middle, p_last,
         author_id AS anchor_id,
         orcid     AS anchor_orcid,
         ARRAY_DISTINCT(CONCAT(lki_ids, aff_ids)) AS anchor_inst
  FROM group_profiles
  WHERE has_orcid
),
straggler AS (
  SELECT p_first, p_middle, p_last,
         author_id AS straggler_id,
         ARRAY_DISTINCT(CONCAT(lki_ids, aff_ids)) AS straggler_inst
  FROM group_profiles
  WHERE NOT has_orcid
),
pair_inst AS (
  SELECT a.p_first, a.p_middle, a.p_last,
         a.anchor_id, s.straggler_id,
         (SIZE(ARRAY_INTERSECT(a.anchor_inst, s.straggler_inst)) > 0) AS inst_overlap
  FROM anchor a
  JOIN straggler s
    ON a.p_first = s.p_first AND a.p_middle = s.p_middle AND a.p_last = s.p_last
),
group_member_coauthors AS (
  -- Per profile, the set of distinct coauthor author_ids on its work_authors rows.
  SELECT wa1.author_id AS profile_id, wa2.author_id AS coauthor_id
  FROM openalex.works.work_authors wa1
  JOIN openalex.works.work_authors wa2
    ON wa2.work_id = wa1.work_id
   AND wa2.author_id <> wa1.author_id
  WHERE wa1.author_id IN (SELECT anchor_id    FROM anchor)
     OR wa1.author_id IN (SELECT straggler_id FROM straggler)
  GROUP BY wa1.author_id, wa2.author_id
),
pair_coauthor AS (
  -- Coauthors shared between anchor and straggler, EXCLUDING any other group member
  -- (otherwise transitive in-group coauthorship inflates the signal artificially).
  SELECT p.p_first, p.p_middle, p.p_last,
         p.anchor_id, p.straggler_id,
         COUNT(DISTINCT ac.coauthor_id) AS n_shared_coauthors
  FROM pair_inst p
  JOIN group_member_coauthors ac ON ac.profile_id = p.anchor_id
  JOIN group_member_coauthors sc ON sc.profile_id = p.straggler_id
                                AND sc.coauthor_id = ac.coauthor_id
  WHERE ac.coauthor_id NOT IN (SELECT anchor_id    FROM anchor)
    AND ac.coauthor_id NOT IN (SELECT straggler_id FROM straggler)
  GROUP BY p.p_first, p.p_middle, p.p_last, p.anchor_id, p.straggler_id
),
pair_signal AS (
  SELECT pi.p_first, pi.p_middle, pi.p_last,
         pi.anchor_id, pi.straggler_id,
         pi.inst_overlap,
         COALESCE(pc.n_shared_coauthors, 0) AS n_shared_coauthors,
         (pi.inst_overlap OR COALESCE(pc.n_shared_coauthors, 0) >= 1) AS any_signal
  FROM pair_inst pi
  LEFT JOIN pair_coauthor pc
    ON pc.anchor_id = pi.anchor_id AND pc.straggler_id = pi.straggler_id
),
group_signal_ok AS (
  -- Keep groups where EVERY straggler has at least one signal vs the anchor.
  SELECT p_first, p_middle, p_last, anchor_id
  FROM pair_signal
  GROUP BY p_first, p_middle, p_last, anchor_id
  HAVING COUNT(*) = COUNT(CASE WHEN any_signal THEN 1 END)
)
SELECT
  ps.p_first, ps.p_middle, ps.p_last,
  ps.anchor_id,
  a.anchor_orcid,
  ps.straggler_id,
  ps.inst_overlap,
  ps.n_shared_coauthors,
  -- Anchor metadata (one row per pair carries it)
  ap.full_name      AS anchor_full_name,
  ap.works_count    AS anchor_works_count,
  ap.cited_by_count AS anchor_cited_by_count,
  ap.created_date   AS anchor_created_date,
  ap.block_key      AS anchor_block_key,
  -- Straggler (loser) metadata
  sp.full_name      AS straggler_full_name,
  sp.orcid          AS straggler_orcid,
  sp.works_count    AS straggler_works_count,
  sp.cited_by_count AS straggler_cited_by_count,
  sp.created_date   AS straggler_created_date,
  sp.block_key      AS straggler_block_key
FROM pair_signal ps
JOIN group_signal_ok ok
  ON ok.p_first = ps.p_first AND ok.p_middle = ps.p_middle
 AND ok.p_last  = ps.p_last  AND ok.anchor_id = ps.anchor_id
JOIN anchor a
  ON a.p_first = ps.p_first AND a.p_middle = ps.p_middle
 AND a.p_last  = ps.p_last  AND a.anchor_id = ps.anchor_id
JOIN group_profiles ap ON ap.author_id = ps.anchor_id
JOIN group_profiles sp ON sp.author_id = ps.straggler_id


### Cell 2: Validation statistics

In [ ]:
%sql
SELECT
  COUNT(DISTINCT anchor_id)                                                      AS n_groups,
  COUNT(*)                                                                       AS n_losers,
  COUNT(DISTINCT anchor_id, straggler_id)                                        AS n_pairs,
  AVG(group_size)                                                                AS avg_group_size,
  PERCENTILE_APPROX(group_size, ARRAY(0.5, 0.95, 0.99))                          AS group_size_pctiles,
  SUM(straggler_works_count)                                                     AS works_to_rewrite,
  COUNT(CASE WHEN inst_overlap THEN 1 END)                                       AS pairs_with_inst,
  COUNT(CASE WHEN n_shared_coauthors >= 1 THEN 1 END)                            AS pairs_with_coauthor,
  COUNT(CASE WHEN inst_overlap AND n_shared_coauthors >= 1 THEN 1 END)           AS pairs_with_both,
  COUNT(CASE WHEN straggler_orcid IS NOT NULL AND straggler_orcid <> ''
              THEN 1 END)                                                        AS losers_with_orcid_unexpected,
  PERCENTILE_APPROX(n_shared_coauthors, ARRAY(0.5, 0.95, 0.99))                  AS coauthor_pctiles
FROM (
  SELECT *,
    COUNT(*) OVER (PARTITION BY anchor_id) + 1 AS group_size
  FROM openalex.authors.merge_candidates_orcid_anchor_multi
)


### Cell 3: Spot-check sample (manual review)

In [ ]:
%sql
WITH sampled_anchors AS (
  SELECT DISTINCT anchor_id
  FROM openalex.authors.merge_candidates_orcid_anchor_multi
  ORDER BY RAND() LIMIT 30
)
SELECT c.anchor_id, c.anchor_orcid, c.p_first, c.p_middle, c.p_last,
       c.anchor_full_name, c.anchor_works_count, c.anchor_cited_by_count,
       c.straggler_id, c.straggler_full_name,
       c.straggler_works_count, c.straggler_cited_by_count,
       c.inst_overlap, c.n_shared_coauthors
FROM openalex.authors.merge_candidates_orcid_anchor_multi c
JOIN sampled_anchors s ON s.anchor_id = c.anchor_id
ORDER BY c.anchor_id, c.straggler_works_count DESC


### Cell 4: Create audit log

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.authors.author_merge_log_orcid_anchor_multi AS
SELECT
  anchor_id              AS winner_author_id,
  anchor_orcid           AS winner_orcid,
  straggler_id           AS loser_author_id,
  straggler_orcid        AS loser_orcid,
  straggler_full_name    AS loser_full_name,
  straggler_works_count  AS loser_works_count,
  straggler_cited_by_count AS loser_cited_by_count,
  straggler_created_date AS loser_created_date,
  p_first, p_middle, p_last,
  inst_overlap,
  n_shared_coauthors,
  current_timestamp()    AS merge_timestamp
FROM openalex.authors.merge_candidates_orcid_anchor_multi


### Cell 5: Snapshot affected work_ids (capture before MERGE rewrites author_id)

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.authors.author_merge_affected_works_orcid_anchor_multi AS
SELECT DISTINCT wa.work_id
FROM openalex.works.work_authors wa
JOIN openalex.authors.author_merge_log_orcid_anchor_multi log
  ON wa.author_id = log.loser_author_id


### Cell 6: Execute the merge — rewrite work_authors.author_id from losers to winners

In [ ]:
%sql
MERGE INTO openalex.works.work_authors AS target
USING (
  SELECT loser_author_id, winner_author_id
  FROM openalex.authors.author_merge_log_orcid_anchor_multi
) AS source
ON target.author_id = source.loser_author_id
WHEN MATCHED THEN
  UPDATE SET
    target.author_id = source.winner_author_id,
    target.updated_at = current_timestamp()


### Cell 7: Queue affected works for reprocessing by UpdateWorkAuthorships

*No DELETE step. Phase 1 reversed its DELETE so CreateAuthors could drain loser `works_count` to 0 on the next refresh; this notebook follows that pattern.*

In [ ]:
%sql
INSERT INTO openalex.institutions.curated_work_ids_pending_sync
SELECT work_id, NULL AS curated_ras, current_timestamp() AS added_datetime
FROM openalex.authors.author_merge_affected_works_orcid_anchor_multi


### Cell 8: Post-merge verification — losers should drain to 0 works after CreateAuthors

In [ ]:
%sql
WITH log AS (
  SELECT loser_author_id, loser_works_count
  FROM openalex.authors.author_merge_log_orcid_anchor_multi
),
loser_state AS (
  SELECT l.loser_author_id, l.loser_works_count,
         a.works_count AS current_works_count
  FROM log l
  JOIN openalex.authors.openalex_authors a ON a.id = l.loser_author_id
)
SELECT
  COUNT(*)                                                       AS total_losers,
  COUNT(CASE WHEN current_works_count = 0 THEN 1 END)            AS losers_drained_to_zero,
  COUNT(CASE WHEN current_works_count > 0 THEN 1 END)            AS losers_still_nonzero,
  PERCENTILE_APPROX(current_works_count, ARRAY(0.5, 0.95, 0.99)) AS nonzero_pctiles,
  MAX(current_works_count)                                       AS max_remaining_works
FROM loser_state
